# Tahap 4: Case Solution Reuse
**Domain:** Pidana Umum - Pemalsuan (Pasal 263 & 266 KUHP)

Notebook ini mencakup:
1. Load dataset dan model dari tahap 3
2. Bangun case solutions dictionary
3. Algoritma majority vote dan weighted similarity
4. Fungsi `predict_outcome()`
5. Demo manual 5 kasus baru
6. Simpan predictions.csv

## Setup

In [1]:
import os

BASE_DIR = r'C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan'

PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
EVAL_DIR      = os.path.join(BASE_DIR, 'data', 'eval')
MODEL_DIR     = os.path.join(BASE_DIR, 'models')
RESULTS_DIR   = os.path.join(BASE_DIR, 'data', 'results')

os.makedirs(RESULTS_DIR, exist_ok=True)

## 1. Install & Import

In [2]:
%pip install transformers torch scikit-learn -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
import pickle
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
import torch
from transformers import AutoTokenizer, AutoModel
from typing import List, Tuple, Dict

## 2. Load Dataset & Models dari Tahap 3


In [4]:
with open(os.path.join(MODEL_DIR, 'train_df.pkl'), 'rb') as f:
    train_df = pickle.load(f)

with open(os.path.join(MODEL_DIR, 'test_df.pkl'), 'rb') as f:
    test_df = pickle.load(f)

with open(os.path.join(MODEL_DIR, 'tfidf_retriever.pkl'), 'rb') as f:
    tfidf_retriever = pickle.load(f)

with open(os.path.join(MODEL_DIR, 'train_tfidf_matrix.pkl'), 'rb') as f:
    train_tfidf = pickle.load(f)

with open(os.path.join(MODEL_DIR, 'svm_pipeline.pkl'), 'rb') as f:
    svm_pipeline = pickle.load(f)

train_embeddings = np.load(os.path.join(MODEL_DIR, 'train_embeddings.npy'))

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

tokenizer  = AutoTokenizer.from_pretrained(os.path.join(MODEL_DIR, 'indobert'))
bert_model = AutoModel.from_pretrained(os.path.join(MODEL_DIR, 'indobert'))
bert_model = bert_model.to(device)
bert_model.eval()

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print('Semua model berhasil di-load.')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Train: 28 | Test: 8
Semua model berhasil di-load.


## 3. Bangun Case Solutions Dictionary


In [5]:
case_solutions: Dict[str, dict] = {}

for _, row in train_df.iterrows():
    case_solutions[row['case_id']] = {
        'amar_putusan' : row.get('amar_putusan', ''),
        'label_svm'    : row.get('label_svm', ''),
        'durasi_bulan' : None if pd.isna(row.get('durasi_bulan')) else int(row.get('durasi_bulan')),
        'no_perkara'   : row.get('no_perkara', ''),
    }

print(f'Case solutions: {len(case_solutions)} kasus')
sample_id = list(case_solutions.keys())[0]
print(f'\nContoh ({sample_id}):')
for k, v in case_solutions[sample_id].items():
    print(f'  {k}: {str(v)[:100]}')

Case solutions: 28 kasus

Contoh (case_012):
  amar_putusan: bebas
  label_svm: bebas
  durasi_bulan: None
  no_perkara: 1382/pid.b/2020/pn jkt.utr


## 4. Algoritma Prediksi

In [6]:
def majority_vote(top_k_results: List[Tuple[str, float]]) -> str:
    labels = [
        case_solutions[cid]['label_svm']
        for cid, _ in top_k_results
        if cid in case_solutions
    ]
    if not labels:
        return 'unknown'
    return Counter(labels).most_common(1)[0][0]


def weighted_vote(top_k_results: List[Tuple[str, float]]) -> str:
    weights: Dict[str, float] = {}
    for cid, score in top_k_results:
        if cid not in case_solutions:
            continue
        label = case_solutions[cid]['label_svm']
        weights[label] = weights.get(label, 0.0) + score
    if not weights:
        return 'unknown'
    return max(weights, key=weights.get)


def get_solution_summary(top_k_results: List[Tuple[str, float]]) -> str:
    if not top_k_results:
        return ''
    best_cid = top_k_results[0][0]
    return case_solutions.get(best_cid, {}).get('amar_putusan', '')

## 5. Fungsi Retrieval dan Predict

In [7]:
def get_bert_embedding(text: str, max_length: int = 512) -> np.ndarray:
    inputs = tokenizer(
        text, return_tensors='pt',
        max_length=max_length, truncation=True, padding=True
    ).to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    attention_mask      = inputs['attention_mask']
    token_embeddings    = outputs.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    embedding = torch.sum(token_embeddings * input_mask_expanded, 1)
    embedding = embedding / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return embedding.cpu().numpy()[0]

def retrieve_tfidf(query: str, k: int = 5) -> List[Tuple[str, float]]:
    query_vec    = tfidf_retriever.transform([query.lower().strip()])
    similarities = cosine_similarity(query_vec, train_tfidf).flatten()
    top_k_idx    = similarities.argsort()[::-1][:k]
    return [(train_df.iloc[i]['case_id'], float(similarities[i])) for i in top_k_idx]

def retrieve_bert(query: str, k: int = 5) -> List[Tuple[str, float]]:
    query_emb    = get_bert_embedding(query.lower().strip()).reshape(1, -1)
    similarities = cosine_similarity(query_emb, train_embeddings).flatten()
    top_k_idx    = similarities.argsort()[::-1][:k]
    return [(train_df.iloc[i]['case_id'], float(similarities[i])) for i in top_k_idx]

def retrieve(query: str, k: int = 5, method: str = 'bert') -> List[Tuple[str, float]]:
    if method == 'tfidf':
        return retrieve_tfidf(query, k)
    return retrieve_bert(query, k)

def predict_outcome(
    query: str,
    k: int = 5,
    method: str = 'bert',
    voting: str = 'weighted'
) -> dict:

    top_k = retrieve(query, k=k, method=method)

    if voting == 'majority':
        predicted_label = majority_vote(top_k)
    else:
        predicted_label = weighted_vote(top_k)

    solution_summary = get_solution_summary(top_k)

    top_k_detail = [
        {
            'case_id'      : cid,
            'similarity'   : round(score, 4),
            'label_svm'    : case_solutions.get(cid, {}).get('label_svm', ''),
            'durasi_bulan' : case_solutions.get(cid, {}).get('durasi_bulan', ''),
            'no_perkara'   : case_solutions.get(cid, {}).get('no_perkara', ''),
        }
        for cid, score in top_k
    ]

    return {
        'predicted_label'  : predicted_label,
        'solution_summary' : solution_summary,
        'top_k_cases'      : top_k_detail,
    }

## 6. Demo Manual — 5 Kasus dari Test Set


In [8]:
demo_cases = test_df.head(5)

for _, row in demo_cases.iterrows():
    print(f"{'='*65}")
    print(f"Query     : {row['case_id']} | {row.get('no_perkara', '-')}")
    print(f"Aktual    : {row['label_svm']} ({row.get('durasi_bulan', '-')} bulan)")

    result = predict_outcome(row['text_full'], k=5, method='bert', voting='weighted')

    print(f"Prediksi  : {result['predicted_label']}")
    print(f"Benar?    : {'YES' if result['predicted_label'] == row['label_svm'] else 'NO'}")
    print(f"Top-5 kasus mirip:")
    for c in result['top_k_cases']:
        print(f"  {c['case_id']} | sim: {c['similarity']} | "
              f"{c['label_svm']} ({c['durasi_bulan']} bln) | {c['no_perkara']}")
    print(f"Amar referensi: {str(result['solution_summary'])[:150]}...")
    print()

Query     : case_009 | 1298/pid.b/2019/pn jkt.utr.
Aktual    : pasal 263 ayat (1) (24.0 bulan)
Prediksi  : pasal 263 ayat (1)
Benar?    : YES
Top-5 kasus mirip:
  case_007 | sim: 0.8787 | pasal 263 ayat (1) (18 bln) | 1204/pid.b/2021/pn jkt.utr
  case_016 | sim: 0.8768 | pasal 266 (6 bln) | 214/pid.b/2020/pn jkt.utr
  case_029 | sim: 0.8731 | pasal 263 ayat (1) (8 bln) | 695/pid.b/2018/pn jkt.utr.
  case_001 | sim: 0.8714 | pasal 266 (12 bln) | 1014/pid.b/2019/pn jkt.utr
  case_030 | sim: 0.8672 | pasal 263 ayat (1) (8 bln) | 697/pid.b/2018/pn jkt.utr.
Amar referensi: bersalah...

Query     : case_018 | 216/pid.b/2020/pn jkt.utr
Aktual    : pasal 266 (16.0 bulan)
Prediksi  : pasal 266
Benar?    : YES
Top-5 kasus mirip:
  case_016 | sim: 0.8934 | pasal 266 (6 bln) | 214/pid.b/2020/pn jkt.utr
  case_017 | sim: 0.8876 | pasal 266 (16 bln) | 215/pid.b/2020/pn jkt.utr
  case_008 | sim: 0.8735 | pasal 263 ayat (2) (12 bln) | 124/pid.b/2020/pn.jkt.utr
  case_001 | sim: 0.8678 | pasal 266 (12 

## 7. Prediksi Seluruh Test Set & Simpan predictions.csv

In [9]:
predictions = []

for _, row in test_df.iterrows():
    result_bert   = predict_outcome(row['text_full'], k=5, method='bert',  voting='weighted')
    result_tfidf  = predict_outcome(row['text_full'], k=5, method='tfidf', voting='weighted')
    svm_pred      = svm_pipeline.predict([row['text_full']])[0]

    top5_ids = [c['case_id'] for c in result_bert['top_k_cases']]

    predictions.append({
        'query_id'              : row['case_id'],
        'no_perkara'            : row.get('no_perkara', ''),
        'true_label'            : row['label_svm'],
        'true_durasi_bulan'     : row.get('durasi_bulan', ''),
        'pred_bert_weighted'    : result_bert['predicted_label'],
        'pred_tfidf_weighted'   : result_tfidf['predicted_label'],
        'pred_svm'              : svm_pred,
        'predicted_solution'    : result_bert['solution_summary'][:300] if result_bert['solution_summary'] else '',
        'top_5_case_ids'        : ', '.join(top5_ids),
    })

pred_df = pd.DataFrame(predictions)

print('=== Akurasi Prediksi pada Test Set ===')
for col in ['pred_bert_weighted', 'pred_tfidf_weighted', 'pred_svm']:
    acc = (pred_df['true_label'] == pred_df[col]).mean()
    print(f'  {col:<25}: {acc:.4f}')

PRED_PATH = os.path.join(RESULTS_DIR, 'predictions.csv')
pred_df.to_csv(PRED_PATH, index=False, encoding='utf-8')
print(f'\npredictions.csv disimpan: {PRED_PATH}')
print(pred_df[['query_id', 'true_label', 'pred_bert_weighted',
               'pred_tfidf_weighted', 'pred_svm']].to_string(index=False))

=== Akurasi Prediksi pada Test Set ===
  pred_bert_weighted       : 0.5000
  pred_tfidf_weighted      : 0.8750
  pred_svm                 : 0.8750

predictions.csv disimpan: C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\data\results\predictions.csv
query_id         true_label pred_bert_weighted pred_tfidf_weighted           pred_svm
case_009 pasal 263 ayat (1) pasal 263 ayat (1)  pasal 263 ayat (1) pasal 263 ayat (1)
case_018          pasal 266          pasal 266           pasal 266          pasal 266
case_034              bebas pasal 263 ayat (1)               bebas              bebas
case_025              bebas          pasal 266               bebas              bebas
case_010 pasal 263 ayat (1) pasal 263 ayat (1)  pasal 263 ayat (1) pasal 263 ayat (1)
case_020 pasal 263 ayat (1)          pasal 266  pasal 263 ayat (1) pasal 263 ayat (1)
case_032 pasal 263 ayat (1) pasal 263 ayat (1)  pasal 263 ayat (1) pasal 263 ayat (1)
case_015 pasal 263 ayat (2) pasal 263 ayat (1)  pasa